# Iteris — Week 1: CAMUS Ingestion + Attention Residual U-Net Baseline
**Project:** DRL-based Automated Medical Image Segmentation  
**Dataset:** CAMUS (Cardiac Acquisitions for Multi-structure Ultrasound Segmentation)  
**Target:** Dice ≥ 0.85 on LV Endocardium  

### To run on Kaggle:
1. Add dataset: `xiaoweixumedicalai/camus` (or your uploaded CAMUS dataset)
2. Enable GPU accelerator (T4 x2 or P100)
3. Run all cells

### To switch to CHAOS: change `CFG = CAMUS_CFG` → `CFG = CHAOS_CFG` in Section 1. Nothing else changes.

## 0 · Install & Imports

In [ ]:
%%capture
!pip install SimpleITK nibabel scikit-image albumentations -q

In [ ]:
import os, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import SimpleITK as sitk
from skimage.transform import resize
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')
print('PyTorch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1 · Configuration
> Switch `CFG = CAMUS_CFG` ↔ `CFG = CHAOS_CFG` to change dataset. Model code is identical.

In [ ]:
CAMUS_CFG = dict(
    dataset       = 'CAMUS',
    modality      = 'ultrasound',
    # ── Kaggle path: add dataset xiaoweixumedicalai/camus ──
    data_root     = '/kaggle/input/camus/training',
    image_size    = 256,
    num_classes   = 4,           # 0=bg  1=LV_endo  2=LV_epi  3=LA
    class_names   = ['background', 'LV_endo', 'LV_epi', 'LA'],
    class_colors  = ['#000000', '#00C9A7', '#F59E0B', '#F87171'],
    views         = ['2CH', '4CH'],
    phases        = ['ED', 'ES'],
    normalize     = 'minmax',    # ultrasound: min-max per image
    label_frac    = 1.0,         # 0.1 / 0.2 / 1.0 for few-shot ablations
    val_split     = 0.15,
    test_split    = 0.10,
    # ── training ──
    batch_size    = 8,
    epochs        = 60,
    lr            = 1e-4,
    weight_decay  = 1e-5,
    patience      = 10,          # early stop on val Dice
    seed          = 42,
    checkpoint    = '/kaggle/working/camus_best.pt',
)

CHAOS_CFG = dict(
    dataset       = 'CHAOS',
    modality      = 'ct',        # switch to 'mri' for MRI subset
    data_root     = '/kaggle/input/chaos-dataset',
    image_size    = 256,
    num_classes   = 5,           # 0=bg  1=liver  2=r_kidney  3=l_kidney  4=spleen
    class_names   = ['background', 'liver', 'right_kidney', 'left_kidney', 'spleen'],
    class_colors  = ['#000000', '#818CF8', '#34D399', '#34D399', '#FB923C'],
    normalize     = 'hu',        # CT: HU windowing before normalisation
    hu_window     = dict(width=150, level=70),
    label_frac    = 1.0,
    val_split     = 0.15,
    test_split    = 0.10,
    batch_size    = 8,
    epochs        = 60,
    lr            = 1e-4,
    weight_decay  = 1e-5,
    patience      = 10,
    seed          = 42,
    checkpoint    = '/kaggle/working/chaos_best.pt',
)

# ─── ACTIVE CONFIG ───────────────────────────────────────────────────────────
CFG = CAMUS_CFG
# ─────────────────────────────────────────────────────────────────────────────

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print(f"Running: {CFG['dataset']} | {CFG['num_classes']} classes | {CFG['image_size']}px")

## 2 · Ingestion
Loads `.mhd` files for CAMUS. CHAOS loader is included but inactive until CFG is switched.

In [ ]:
def load_mhd(path: Path) -> np.ndarray:
    """Read a .mhd/.raw pair and return a 2-D float32 array."""
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    return arr[0] if arr.ndim == 3 else arr   # drop z if present


def ingest_camus(data_root: str, views, phases) -> list:
    """Walk CAMUS training folder → list of dicts with image/mask arrays."""
    root = Path(data_root)
    samples = []
    patient_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
    print(f'Found {len(patient_dirs)} patients')
    for pd in tqdm(patient_dirs, desc='Ingesting CAMUS'):
        pid = pd.name
        for view in views:
            for phase in phases:
                img_p = pd / f'{pid}_{view}_{phase}.mhd'
                gt_p  = pd / f'{pid}_{view}_{phase}_gt.mhd'
                if img_p.exists() and gt_p.exists():
                    samples.append(dict(
                        patient = pid,
                        view    = view,
                        phase   = phase,
                        image   = load_mhd(img_p),
                        mask    = load_mhd(gt_p).astype(np.int64),
                    ))
    print(f'Total samples: {len(samples)}')
    return samples


def ingest_chaos_ct(data_root: str) -> list:
    """
    CHAOS CT: folder structure → CТ/{patient_id}/DICOM_anon/*.dcm
    Ground truth  → CT/{patient_id}/Ground/*.png  (labels 55/110/155/200)
    Inactive in Week 1; activated when CFG = CHAOS_CFG.
    """
    import pydicom
    from PIL import Image
    LABEL_MAP = {55: 1, 110: 2, 155: 3, 200: 4}  # liver/r_kidney/l_kidney/spleen
    root = Path(data_root) / 'CT'
    samples = []
    for patient_dir in sorted(root.iterdir()):
        dcm_dir = patient_dir / 'DICOM_anon'
        gt_dir  = patient_dir / 'Ground'
        if not dcm_dir.exists(): continue
        dcm_files = sorted(dcm_dir.glob('*.dcm'))
        gt_files  = sorted(gt_dir.glob('*.png')) if gt_dir.exists() else []
        for dcm_f, gt_f in zip(dcm_files, gt_files):
            ds    = pydicom.dcmread(str(dcm_f))
            image = ds.pixel_array.astype(np.float32)
            gt    = np.array(Image.open(gt_f))
            mask  = np.zeros_like(gt, dtype=np.int64)
            for px_val, cls_id in LABEL_MAP.items():
                mask[gt == px_val] = cls_id
            samples.append(dict(patient=patient_dir.name, image=image, mask=mask))
    return samples


# ─── Run ingestion ────────────────────────────────────────────────────────────
if CFG['dataset'] == 'CAMUS':
    all_samples = ingest_camus(CFG['data_root'], CFG['views'], CFG['phases'])
elif CFG['dataset'] == 'CHAOS':
    all_samples = ingest_chaos_ct(CFG['data_root'])

print(f"Sample keys: {list(all_samples[0].keys())}")
print(f"Image shape: {all_samples[0]['image'].shape}  |  dtype: {all_samples[0]['image'].dtype}")
print(f"Mask  shape: {all_samples[0]['mask'].shape}   |  unique labels: {np.unique(all_samples[0]['mask'])}")

## 3 · EDA — Sanity Check

In [ ]:
from matplotlib.colors import ListedColormap

def show_samples(samples, n=4, cfg=CFG):
    cmap = ListedColormap(cfg['class_colors'])
    idx  = random.sample(range(len(samples)), min(n, len(samples)))
    fig, axes = plt.subplots(n, 2, figsize=(8, 4*n))
    for row, i in enumerate(idx):
        s = samples[i]
        axes[row, 0].imshow(s['image'], cmap='gray')
        axes[row, 0].set_title(f"{s.get('patient','')} {s.get('view','')} {s.get('phase','')}")
        axes[row, 0].axis('off')
        axes[row, 1].imshow(s['image'], cmap='gray')
        axes[row, 1].imshow(s['mask'], cmap=cmap, alpha=0.5,
                            vmin=0, vmax=cfg['num_classes']-1)
        axes[row, 1].set_title('Overlay')
        axes[row, 1].axis('off')
    patches = [mpatches.Patch(color=c, label=n)
               for c, n in zip(cfg['class_colors'][1:], cfg['class_names'][1:])]
    fig.legend(handles=patches, loc='upper right')
    plt.tight_layout()
    plt.show()

show_samples(all_samples)

# Class distribution
all_labels = np.concatenate([s['mask'].flatten() for s in all_samples])
vals, counts = np.unique(all_labels, return_counts=True)
plt.figure(figsize=(7,3))
plt.bar([CFG['class_names'][int(v)] for v in vals], counts/counts.sum())
plt.ylabel('Fraction of pixels')
plt.title(f'{CFG["dataset"]} — class pixel distribution')
plt.tight_layout()
plt.show()

## 4 · Train / Val / Test Split + Dataset

In [ ]:
# ─── Label-fraction sub-sampling (for few-shot ablations) ────────────────────
if CFG['label_frac'] < 1.0:
    n_keep = max(1, int(len(all_samples) * CFG['label_frac']))
    all_samples = random.sample(all_samples, n_keep)
    print(f'Using {n_keep} samples ({CFG["label_frac"]*100:.0f}%)')

# Split by patient ID to avoid data leakage across views/phases
patients = list({s['patient'] for s in all_samples})
random.shuffle(patients)
n_test  = max(1, int(len(patients) * CFG['test_split']))
n_val   = max(1, int(len(patients) * CFG['val_split']))
test_pts  = set(patients[:n_test])
val_pts   = set(patients[n_test:n_test+n_val])
train_pts = set(patients[n_test+n_val:])

train_s = [s for s in all_samples if s['patient'] in train_pts]
val_s   = [s for s in all_samples if s['patient'] in val_pts]
test_s  = [s for s in all_samples if s['patient'] in test_pts]
print(f'Train: {len(train_s)}  Val: {len(val_s)}  Test: {len(test_s)}')

In [ ]:
def normalize_image(image: np.ndarray, cfg: dict) -> np.ndarray:
    """Modality-aware normalisation. Keeps the model input in [0,1]."""
    mode = cfg['normalize']
    if mode == 'minmax':
        mn, mx = image.min(), image.max()
        return ((image - mn) / (mx - mn + 1e-8)).astype(np.float32)
    if mode == 'zscore':
        return ((image - image.mean()) / (image.std() + 1e-8)).astype(np.float32)
    if mode == 'hu':
        w, l = cfg['hu_window']['width'], cfg['hu_window']['level']
        lo, hi = l - w/2, l + w/2
        image = np.clip(image, lo, hi)
        return ((image - lo) / w).astype(np.float32)
    raise ValueError(f'Unknown normalisation mode: {mode}')


TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(0.001, 0.005), p=0.3),
])


class MedSegDataset(Dataset):
    """
    Generic medical segmentation dataset.
    Identical interface for CAMUS, CHAOS, ACDC, ISIC, DRIVE.
    Only `cfg` changes between datasets.
    """
    def __init__(self, samples: list, cfg: dict, augment: bool = False):
        self.samples = samples
        self.cfg     = cfg
        self.sz      = cfg['image_size']
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s     = self.samples[idx]
        image = resize(s['image'], (self.sz, self.sz),
                       preserve_range=True, anti_aliasing=True).astype(np.float32)
        mask  = resize(s['mask'],  (self.sz, self.sz),
                       order=0, preserve_range=True, anti_aliasing=False).astype(np.int64)
        image = normalize_image(image, self.cfg)

        if self.augment:
            aug   = TRAIN_AUG(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask']

        image = torch.from_numpy(image).unsqueeze(0).float()  # (1, H, W)
        mask  = torch.from_numpy(mask).long()                 # (H, W)
        return image, mask


train_ds = MedSegDataset(train_s, CFG, augment=True)
val_ds   = MedSegDataset(val_s,   CFG, augment=False)
test_ds  = MedSegDataset(test_s,  CFG, augment=False)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# Quick shape check
imgs, masks = next(iter(train_dl))
print(f'Batch  — images: {imgs.shape}  masks: {masks.shape}  dtype: {imgs.dtype}')
print(f'Mask   — unique classes in batch: {masks.unique().tolist()}')

## 5 · Model — Attention Residual U-Net
ResNet-34 style encoder → attention-gated skip connections → transposed-conv decoder.  
`in_channels` and `num_classes` are the only wiring points that differ between datasets.

In [ ]:
class ResBlock(nn.Module):
    """Two-conv residual block with optional projection shortcut."""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.skip  = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )
    def forward(self, x):
        return self.relu(
            self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.skip(x)
        )


class AttentionGate(nn.Module):
    """
    Soft attention gate (Oktay et al., 2018).
    g = gating signal from decoder; x = skip feature from encoder.
    """
    def __init__(self, F_g: int, F_x: int, F_int: int):
        super().__init__()
        self.W_g   = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x   = nn.Sequential(nn.Conv2d(F_x, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi   = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g_up = F.interpolate(self.W_g(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        alpha = self.psi(self.relu(g_up + self.W_x(x)))
        return x * alpha


class AttentionResUNet(nn.Module):
    """
    Attention Residual U-Net — dataset-agnostic.

    Parameters
    ----------
    in_channels : 1 for greyscale (CAMUS/CHAOS/ACDC/DRIVE/ISIC)
                  4 for BraTS multi-param MRI
    num_classes : from CFG['num_classes']
    features    : channel widths at each encoder scale
    """
    def __init__(
        self,
        in_channels: int = 1,
        num_classes: int = 4,
        features: list  = [64, 128, 256, 512],
    ):
        super().__init__()
        F = features
        self.pool = nn.MaxPool2d(2)

        # Encoder
        self.enc1 = ResBlock(in_channels, F[0])
        self.enc2 = ResBlock(F[0], F[1])
        self.enc3 = ResBlock(F[1], F[2])
        self.enc4 = ResBlock(F[2], F[3])

        # Bottleneck
        self.bottleneck = ResBlock(F[3], F[3]*2)

        # Attention gates  (g_channels, x_channels, intermediate)
        self.att4 = AttentionGate(F[3]*2, F[3], F[3]//2)
        self.att3 = AttentionGate(F[3],   F[2], F[2]//2)
        self.att2 = AttentionGate(F[2],   F[1], F[1]//2)
        self.att1 = AttentionGate(F[1],   F[0], F[0]//2)

        # Decoder
        self.up4  = nn.ConvTranspose2d(F[3]*2, F[3], kernel_size=2, stride=2)
        self.dec4 = ResBlock(F[3]*2, F[3])

        self.up3  = nn.ConvTranspose2d(F[3], F[2], kernel_size=2, stride=2)
        self.dec3 = ResBlock(F[2]*2, F[2])

        self.up2  = nn.ConvTranspose2d(F[2], F[1], kernel_size=2, stride=2)
        self.dec2 = ResBlock(F[1]*2, F[1])

        self.up1  = nn.ConvTranspose2d(F[1], F[0], kernel_size=2, stride=2)
        self.dec1 = ResBlock(F[0]*2, F[0])

        self.head = nn.Conv2d(F[0], num_classes, kernel_size=1)

    def forward(self, x):
        # ── Encoder ──────────────────────────────────────────────────────────
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        # ── Decoder with attention ────────────────────────────────────────────
        d4 = self.dec4(torch.cat([self.up4(b),  self.att4(b,  e4)], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), self.att3(d4, e3)], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), self.att2(d3, e2)], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), self.att1(d2, e1)], dim=1))

        return self.head(d1)


model = AttentionResUNet(
    in_channels = 1,
    num_classes = CFG['num_classes'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params/1e6:.2f}M')

# Dry run
with torch.no_grad():
    dummy = torch.zeros(2, 1, CFG['image_size'], CFG['image_size']).to(DEVICE)
    out   = model(dummy)
print(f'Output shape: {out.shape}  (expect [2, {CFG["num_classes"]}, {CFG["image_size"]}, {CFG["image_size"]}])')

## 6 · Loss & Metrics

In [ ]:
class DiceLoss(nn.Module):
    """Soft multi-class Dice loss (excludes background class 0)."""
    def __init__(self, smooth: float = 1e-6, ignore_bg: bool = True):
        super().__init__()
        self.smooth    = smooth
        self.ignore_bg = ignore_bg

    def forward(self, logits, targets):
        C     = logits.shape[1]
        probs = F.softmax(logits, dim=1)                                  # (B, C, H, W)
        oh    = F.one_hot(targets, C).permute(0, 3, 1, 2).float()        # (B, C, H, W)
        start = 1 if self.ignore_bg else 0
        inter = (probs[:, start:] * oh[:, start:]).sum(dim=(0, 2, 3))
        denom = (probs[:, start:] + oh[:, start:]).sum(dim=(0, 2, 3))
        dice  = (2 * inter + self.smooth) / (denom + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    """0.5 * Dice + 0.5 * CrossEntropy — standard for medical segmentation."""
    def __init__(self, alpha: float = 0.5):
        super().__init__()
        self.dice  = DiceLoss()
        self.ce    = nn.CrossEntropyLoss()
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1 - self.alpha) * self.ce(logits, targets)


@torch.no_grad()
def compute_metrics(pred: torch.Tensor, gt: torch.Tensor, num_classes: int, smooth: float = 1e-6) -> dict:
    """
    Compute per-class Dice and IoU, plus mean over foreground classes.
    pred, gt : (H, W) integer tensors.
    """
    dice_vals, iou_vals = [], []
    for c in range(1, num_classes):      # skip background
        p = (pred == c).float()
        g = (gt   == c).float()
        inter = (p * g).sum()
        dice  = (2 * inter + smooth) / (p.sum() + g.sum() + smooth)
        iou   = (inter + smooth) / (p.sum() + g.sum() - inter + smooth)
        dice_vals.append(dice.item())
        iou_vals.append(iou.item())

    out = {f'dice_c{c}': dice_vals[c-1] for c in range(1, num_classes)}
    out.update({f'iou_c{c}': iou_vals[c-1] for c in range(1, num_classes)})
    out['dice_mean'] = float(np.mean(dice_vals))
    out['iou_mean']  = float(np.mean(iou_vals))
    return out


criterion = CombinedLoss(alpha=0.5)
print('Loss and metrics ready.')

## 7 · Training Loop

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6
)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss, all_m = 0.0, []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        for b in range(len(preds)):
            all_m.append(compute_metrics(preds[b], masks[b], num_classes))
    avg_loss = total_loss / len(loader)
    avg_m    = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    return avg_loss, avg_m


# ─── Training ─────────────────────────────────────────────────────────────────
best_dice   = 0.0
patience_ct = 0
history     = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}

print(f'Starting training — {CFG["epochs"]} epochs | device: {DEVICE}')
for epoch in range(1, CFG['epochs'] + 1):
    tr_loss          = train_epoch(model, train_dl, optimizer, criterion, DEVICE)
    vl_loss, vl_m    = eval_epoch(model, val_dl, criterion, DEVICE, CFG['num_classes'])
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_dice'].append(vl_m['dice_mean'])
    history['val_iou'].append(vl_m['iou_mean'])

    improved = vl_m['dice_mean'] > best_dice
    if improved:
        best_dice   = vl_m['dice_mean']
        patience_ct = 0
        torch.save(model.state_dict(), CFG['checkpoint'])
    else:
        patience_ct += 1

    marker = '✓' if improved else ' '
    print(f'Ep {epoch:03d}/{CFG["epochs"]} | '
          f'tr_loss {tr_loss:.4f} | '
          f'vl_loss {vl_loss:.4f} | '
          f'Dice {vl_m["dice_mean"]:.4f} | '
          f'IoU {vl_m["iou_mean"]:.4f} {marker}')

    if patience_ct >= CFG['patience']:
        print(f'Early stop at epoch {epoch}.')
        break

print(f'Best val Dice: {best_dice:.4f}')

## 8 · Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train loss')
ax1.plot(history['val_loss'],   label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss curves'); ax1.legend()

ax2.plot(history['val_dice'], label='Val Dice')
ax2.plot(history['val_iou'],  label='Val IoU')
ax2.axhline(0.85, color='red', linestyle='--', label='Target Dice 0.85')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.set_title(f'{CFG["dataset"]} — Dice / IoU'); ax2.legend()

plt.suptitle(f'{CFG["dataset"]} Attention Residual U-Net — Week 1 Baseline')
plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=150)
plt.show()

## 9 · Test-set Evaluation

In [ ]:
model.load_state_dict(torch.load(CFG['checkpoint'], map_location=DEVICE))
_, test_m = eval_epoch(model, test_dl, criterion, DEVICE, CFG['num_classes'])

print('\n── Test Results ────────────────────────────────')
print(f'  Mean Dice : {test_m["dice_mean"]:.4f}')
print(f'  Mean IoU  : {test_m["iou_mean"]:.4f}')
print('  Per-class Dice:')
for c in range(1, CFG['num_classes']):
    print(f'    {CFG["class_names"][c]:15s}: {test_m[f"dice_c{c}"]:.4f}')
print('────────────────────────────────────────────────')
print(f'  Target (LV_endo Dice ≥ 0.85): '
      f'{"PASS ✓" if test_m["dice_c1"] >= 0.85 else "not yet — keep training"}')

## 10 · Qualitative Visualisation

In [ ]:
model.eval()
cmap = ListedColormap(CFG['class_colors'])
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4*n_show))

subset = random.sample(test_s, n_show)
for row, s in enumerate(subset):
    img_t, mask_t = test_ds[test_s.index(s)]
    with torch.no_grad():
        pred = model(img_t.unsqueeze(0).to(DEVICE)).argmax(dim=1).squeeze().cpu().numpy()
    gt   = mask_t.numpy()
    img  = img_t.squeeze().numpy()

    axes[row,0].imshow(img, cmap='gray');   axes[row,0].set_title('Input');       axes[row,0].axis('off')
    axes[row,1].imshow(img, cmap='gray')
    axes[row,1].imshow(gt, cmap=cmap, alpha=0.55, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
    axes[row,2].imshow(img, cmap='gray')
    axes[row,2].imshow(pred, cmap=cmap, alpha=0.55, vmin=0, vmax=CFG['num_classes']-1)
    axes[row,2].set_title('Prediction');   axes[row,2].axis('off')

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CFG['class_colors'][1:], CFG['class_names'][1:])]
fig.legend(handles=patches, loc='upper right')
plt.suptitle(f'{CFG["dataset"]} — Qualitative Results')
plt.tight_layout()
plt.savefig('/kaggle/working/qualitative_results.png', dpi=150)
plt.show()

## 11 · Save Checkpoint Info
Saves a JSON summary so the DRL pipeline in Week 2 can load the correct baseline checkpoint.

In [ ]:
import json
summary = {
    'dataset'      : CFG['dataset'],
    'model'        : 'AttentionResUNet',
    'num_classes'  : CFG['num_classes'],
    'image_size'   : CFG['image_size'],
    'checkpoint'   : CFG['checkpoint'],
    'best_val_dice': round(best_dice, 4),
    'test_dice'    : {CFG['class_names'][c]: round(test_m[f'dice_c{c}'], 4)
                      for c in range(1, CFG['num_classes'])},
    'test_dice_mean': round(test_m['dice_mean'], 4),
    'test_iou_mean' : round(test_m['iou_mean'],  4),
}
out_path = CFG['checkpoint'].replace('.pt', '_summary.json')
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))